# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENROUTER_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-oss-120b'
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [8]:
def select_relevant_links(url):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog / news', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'services / curriculum',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills / proficiency',
   'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [11]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-oss-120b
Found 6 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-oss-120b
Found 9 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
meta-models/Muse-Glimmer-30B
Updated
2 days ago
•
121k
•
1.45k
MiniMaxAI/MiniMax-H3
Updated
1 day ago
•
1.61M
•
3.85k
Qwen/Qwen3.8-2.4T-A95B
Updated
2 days ago
•
1.01k
•
827
Lightricks/LTX-2.5
Updated
1 day ago
•
57.3k
•
755
deepseek-ai/DeepSeek-V4-Flash-0731
Updated
13 days ago
•
1.

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-oss-120b
Found 10 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nmeta-models/Muse-Glimmer-30B\nUpdated\n3 days ago\n•\n121k\n•\n1.45k\nMiniMaxAI/MiniMax-H3\nUpdated\n1 day ago\n•\n1

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nmeta-models/Muse-Glimmer-30B\nUpdated\n3 days ago\n•\n121k\n•\n1.45k\nMiniMaxAI/MiniMax-H3\nUpdated\n1 day ago\n•\n1.61M\n•\n3.86k\nQwen/Qwen3.8-2.4T-A95B\nUpdated\n2 days ago\n•\n1.01k\n•\n834\nLightricks/LTX-2.5\nUpdated\n1 day ago\n•\n57.3k\n•\n760\ndeepseek-ai/DeepSeek-V4-Flash-0731\nUpdated\n13 days ago\n•\n1.43M\n•\n3.35k\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nMCP\n969\nWan2.2 14B Fast Preview [NEW]\n🏆\n969\nGenerate animated video from a single image with custom prompts\nRunning\non\nZero\nMCP\nFeatured\n2.51k\nQwen-Image-Edit-2511-LoRAs-Fast\n🎃\n2.51k\nDemo of the Collection of Qwen Image Edit LoRAs\nRunning\n218\nFree AI Detector\n🔍\n218\nFree AI detector - AI text generation checker. Lynote.\nRunning\non\nZero\nAgents\nFeatured\n249\nMiniMax H3\n🎬\n249\nVideo generation with a synchronized soundtrack\nRunning\non\nZero\nMCP\n97\nWan2.2 14B Custom Lora\n🏆\n97\nCreate animated videos from a single image\nBrowse 1M+ applications\nDatasets\nHuggingFaceFW/fineweb\nUpdated\nJul 11, 2025\n•\n419k\n•\n3.18k\nAnthropic/hh-rlhf\nUpdated\nMay 26, 2023\n•\n32.9k\n•\n1.96k\nr0b0tlab/qwen3.8-max-glm5.2-kimi-k3-distillation\nUpdated\n12 days ago\n•\n2.12k\n•\n88\nostris/minimax_h3_1k\nUpdated\n4 days ago\n•\n2.33k\n•\n28\nMatrAIx2026/MatrAIx_Persona_1M\nUpdated\n12 days ago\n•\n8.8k\n•\n41\nBrowse 500k+ datasets\nThe Home of Machine Learning\nCreate, discover and collaborate on ML better.\nThe collaboration platform\nHost and collaborate on unlimited public models, datasets and applications.\nMov\n## Relevant Links:\n\n\n### Link: homepage\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nmeta-models/Muse-Glimmer-30B\nUpdated\n3 days ago\n•\n121k\n•\n1.45k\nMiniMaxAI/MiniMax-H3\nUpdated\n1 day ago\n•\n1.61M\n•\n3.86k\nQwen/Qwen3.8-2.4T-A95B\nUpdated\n2 days ago\n•\n1.01k\n•\n834\nLightricks/LTX-2.5\nUpdated\n1 day ago\n•\n57.3k\n•\n760\ndeepseek-ai/DeepSeek-V4-Flash-0731\nUpdated\n13 days ago\n•\n1.43M\n•\n3.35k\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nMCP\n969\nWan2.2 14B Fast Preview [NEW]\n🏆\n969\nGenerate animated video from a single image with custom prompts\nRunning\non\nZero\nMCP\nFeatured\n2.51k\nQwen-Image-Edit-2511-LoRAs-Fast\n🎃\n2.51k\nDemo of the Collection of Qwen Image Edit LoRAs\nRunning\n218\nFree AI Detector\n🔍\n218\nFree AI detector - AI text generation checker. Lynote.\nRunning\non\nZero\nAgents\nFeatured\n249\nMiniMax H3\n🎬\n249\nVideo generation with a synchronized soundtrack\nRunning\non\nZero\nMCP\n97\nWan2.2 14B Custom Lora\n🏆\n97\nCreate animated videos from a single image\nBrowse 1M+ applications\nDatasets\nHuggingFaceFW/fineweb\nUpdated\nJul 11, 2025\n•\n419k\n•\n3.18k\nAnthropic/hh-rlhf\nUpdated\nMay 26, 2023\n•\n32.9k\n•\n1.96k\nr0b0tlab/qwen3.8-max-glm5.2-kimi-k3-distillation\nUpdated\n12 days ago\n•\n2.12k\n•\n88\nostris/minimax_h3_1k\nUpdated\n4 days ago\n•\n2.33k\n•\n28\nMatrAIx2026/MatrAIx_Persona_1M\nUpdated\n12 days ago\n•\n8.8k\n•\n41\nBrowse 500k+ datasets\nThe Home of Machine Learning\nCreate, discover and collaborate on ML better.\nThe collaboration platform\nHost and collaborate on unlimited public models, datasets and applications.\nMov\n\n### Link: about page\nBrand assets - Hugging Face\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nHugging Face\nBrand Assets\nBrand Assets\nDownload the official Hugging Face brand assets to use in your projects.\nHF Logos\n.svg\n.png\n.ai\n.svg\n.png\n.ai\n.svg\n.png\n.ai\n.svg\n.png\n.ai\nHF Colors\n#FFD21E\n#FF9D00\n#6B7280\nHF Bio\n“\nHugging Face is the collaboration platform for the machine learning community.\n\nThe Hugging Face Hub works as a cen'

In [17]:
def create_brochure(company_name, url):
    response = client.chat.completions.create(
        model="gpt-oss-120b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-oss-120b
Found 8 relevant links


**Hugging Face – The AI Community Building the Future**  
*Collaboration, openness, and impact at the heart of every AI breakthrough.*

---  

## 🚀 Who We Are  
Hugging Face is the world’s leading collaboration platform for the machine‑learning ecosystem. With **2 M+ open‑source models**, **500 k+ datasets**, and **1 M+ AI applications (Spaces)**, we empower researchers, developers, and enterprises to create, share, and scale AI responsibly.  

Our mission: **Make state‑of‑the‑art ML accessible to everyone**—from hobbyists building a chatbot in their spare time to Fortune‑500 firms deploying production‑grade inference pipelines.

---  

## 🌐 Core Products & Services  

| Product | What It Does | Who Uses It |
|---|---|---|
| **Models Hub** | Host, version, and discover 2 M+ pre‑trained models (LLMs, vision, audio, multimodal). | Researchers, startups, large enterprises. |
| **Datasets Hub** | Publish and explore 500 k+ curated datasets with built‑in licensing & provenance. | Data scientists, academia, data‑driven product teams. |
| **Spaces** | Ready‑to‑run interactive AI apps (Gradio, Streamlit) that run on a serverless “Zero‑Cost” tier or paid compute. | Creators, educators, product demos. |
| **Buckets** | Secure object storage for training data, model checkpoints, and AI assets. | MLOps teams, AI‑ops platforms. |
| **Enterprise & PRO** | Private workspaces, SLA‑backed inference endpoints, fine‑grained access controls, and dedicated support. | Enterprises needing compliance, security, or scaling guarantees. |
| **Inference Providers & Endpoints** | Managed, low‑latency APIs for any model in the Hub, with usage‑based pricing. | Apps, SaaS platforms, internal tooling. |
| **Learning & Community** | Docs, tutorials, Discord, Forum, Daily Papers, and the new **Hugging Face Fundamentals** learning track. | Students, up‑skillers, community contributors. |

---  

## 🤝 Community & Culture  

- **Open‑source first** – All core libraries (Transformers, Diffusers, Accelerate, etc.) live on GitHub under permissive licenses.  
- **Collaboration‑driven** – Users can fork, improve, and publish models/datasets with a single click; every contribution is tracked and credited.  
- **Inclusive & transparent** – A vibrant Discord, forum, and monthly *Daily Papers* keep the community informed and engaged.  
- **Innovation mindset** – From **HuggingChat** to cutting‑edge multimodal agents, we experiment openly and share results publicly.  
- **Human‑centric values** – Respect, curiosity, and a “we’re all in this together” spirit guide hiring, product decisions, and community moderation.  

---  

## 🏢 Customers & Partners  

| Segment | Representative Users | What They Achieve |
|---|---|---|
| **Tech giants & AI labs** | Meta, Microsoft, Google, Anthropic, DeepSeek | Rapid prototyping of LLMs, sharing research artifacts. |
| **Enterprise software** | Salesforce, UiPath, Snowflake | Secure, private model hosting and scalable inference via Enterprise. |
| **Startups & SaaS** | MiniMax AI, Lightricks, Wan2.2 | Deploy custom AI‑powered products (video generation, image editing) on Spaces. |
| **Academic & research institutes** | Universities worldwide, OpenAI collaborators | Access to massive open datasets and reproducible research pipelines. |
| **Industry verticals** | Healthcare, finance, e‑commerce firms | Leverage domain‑specific models and datasets while staying compliant. |

---  

## 📈 Why Invest in Hugging Face  

- **Network effect** – Every new model or dataset enriches the Hub, attracting more users and generating more data.  
- **Monetization pathways** – PRO subscriptions, Enterprise contracts, inference‑as‑a‑service, and storage Buckets.  
- **Scalable infrastructure** – Cloud‑agnostic inference providers, low‑latency endpoints, and growing partner ecosystem.  
- **Thought leadership** – Regular publications (State of Open‑Source, AI + Compute Landscape) position Hugging Face at the forefront of AI policy and technology discourse.  

---  

## 👩‍💻 Careers & Growth  

- **Talent size** – Over **185** full‑time members across engineering, research, product, and community roles, growing rapidly.  
- **Roles in demand** – Machine‑learning engineers, data‑infrastructure architects, AI safety researchers, product designers, community managers, and sales engineers for Enterprise.  
- **Work environment** – Remote‑first, flexible hours, collaborative code reviews, “hack‑the‑world” days, and generous learning budgets (e.g., access to Hugging Face Fundamentals on DataCamp).  
- **Diversity & Inclusion** – Active recruitment from under‑represented groups, employee resource groups, and a transparent equity‑sharing program.  

*Explore open positions at* **[huggingface.co/careers]** *(link placeholder).*

---  

## 📞 Get In Touch  

- **Website:** https://huggingface.co  
- **Community channels:** Discord, Forum, Twitter, LinkedIn  
- **Support:** Enterprise Support portal, public GitHub issues for open‑source tools  

---  

### Brand Guidelines (quick reference)  

- **Primary colors:** `#FFD21E` (vibrant yellow), `#FF9D00` (orange), `#6B7280` (neutral gray).  
- **Logo assets:** Available in SVG, PNG, and AI formats for both light and dark backgrounds.  

---  

*Hugging Face is more than a platform – it’s a global movement turning curiosity into tangible AI solutions. Join us to build the next generation of intelligent products.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = client.chat.completions.create(
        model="gpt-oss-120b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-oss-120b
Found 10 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face – The AI Community Building the Future  

## Who We Are  
Hugging Face is the world’s fastest‑growing open‑source hub for machine‑learning (ML) collaboration. Our platform brings together **2 M+ models**, **500 k+ datasets**, and **1 M+ ready‑to‑run applications (Spaces)**, enabling developers, researchers, and enterprises to discover, share, and deploy AI solutions at scale.  

## What We Do  

| Pillar | What It Offers | Why It Matters |
|--------|----------------|----------------|
| **Models** | Host, version, and serve unlimited public and private models. Trending models like **Muse‑Glimmer‑30B**, **MiniMax‑H3**, and **Qwen‑3.8‑2.4T** showcase community activity. | Accelerates research and product development by removing friction from model discovery and reuse. |
| **Datasets** | Curated, searchable datasets (e.g., **fineweb**, **hh‑rlhf**) with full provenance and community contributions. | Gives teams high‑quality training data without costly collection pipelines. |
| **Spaces** | Interactive demos and AI‑powered apps that run instantly on Hugging Face hardware (Zero‑cost compute tiers, MCP). | Lets anyone prototype, showcase, and share AI products with a single click. |
| **Buckets** | Scalable cloud storage for large model artefacts, training logs, and data pipelines. | Provides a secure, cost‑effective backbone for enterprise‑grade workloads. |
| **Inference & Enterprise** | Managed inference endpoints, dedicated support, and integration with major cloud providers. | Guarantees low‑latency, production‑ready AI services for mission‑critical use cases. |
| **HuggingChat & PRO** | Conversational AI, premium features, and priority support for power users and businesses. | Drives faster time‑to‑value for internal teams and external customers. |

## Community & Culture  

- **Open‑source at heart** – Every model and dataset is created, reviewed, and improved by a global community of researchers, engineers, and hobbyists.  
- **Collaborative ethos** – Our Hub encourages “fork‑and‑pull” workflows, transparent versioning, and community‑driven benchmarking.  
- **Inclusive branding** – The friendly “🤗” logo, bright brand palette (#FFD21E, #FF9D00, #6B7280), and open‑access documentation reflect our belief that AI should be approachable for everyone.  
- **Learning & networking** – Hugging Face runs a vibrant Discord, forum, blog, and regular “Daily Papers” digests to keep members up‑to‑date on the latest research.  

## Who Uses Hugging Face  

| Segment | Typical Users | Example Use‑Cases |
|---------|---------------|-------------------|
| **Start‑ups & Developers** | Individual engineers, indie AI teams | Rapid prototyping with Spaces, fine‑tuning open‑source models. |
| **Enterprises** | Fortune 500 tech, finance, healthcare, e‑commerce | Private model hosting, managed inference endpoints, compliance‑ready data pipelines. |
| **Research Institutions** | Universities, labs, NGOs | Sharing state‑of‑the‑art models, collaborative dataset curation. |
| **Product Teams** | SaaS platforms, content creators | Embedding AI features (text generation, image editing, video synthesis) directly via APIs. |

> **“Hugging Face is the collaboration platform for the machine learning community.”** – Official brand statement  

## Careers & Talent  

Hugging Face is expanding rapidly and continuously hires across many disciplines:

- **Engineering** – Backend, ML infrastructure, inference scaling, security.  
- **Research** – NLP, vision, multimodal, reinforcement learning.  
- **Product & Design** – UX for Spaces, documentation, community tools.  
- **Customer Success & Enterprise** – Solutions architects, support engineers, partnership managers.  

The **Team & Enterprise** portal lists current openings, and our culture prizes curiosity, openness, and impact. Candidates are encouraged to contribute to open‑source projects as part of the interview process.  

## Getting Started  

1. **Sign up** – Create a free account on the Hub.  
2. **Explore** – Browse 2 M+ models or 500 k+ datasets; run a Space instantly.  
3. **Collaborate** – Fork a model, add a dataset, or launch your own Space.  
4. **Scale** – Upgrade to PRO or Enterprise for private hosting, dedicated support, and enterprise‑grade SLAs.  

## Contact & Resources  

- **Website:** https://huggingface.co  
- **Docs & API:** Comprehensive guides for models, datasets, and inference.  
- **Community:** Discord, Forum, GitHub, and weekly **Daily Papers** newsletter.  
- **Brand Assets:** Official logos (SVG/PNG/AI) and color palette available for partners and media.  

*Join the AI revolution. Build, share, and ship smarter with Hugging Face.*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>